# PageIndex - Vectorless RAG

### Section 1: Key Concepts
- Traditional RAG -> chunk -> embed -> cosine similarity -> retrieve
- PageIndex RAG -> build tree -> LLM reasons over the tree -> retrieve exact sections

The problem with vector RAG is that similarity != relevance

In [27]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

True

In [28]:
from pageindex import PageIndexClient
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
print(llm)

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
print(pi_client)

metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}} output_version=None profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-2.5-flash-lite' client=<google.genai.client.Client object at 0x0000022588F1AE90> default_metadata=() model_kwargs={}


#### Section 2: Upload and Index a PDF

1. Upload you pdf to pageindex cloud
2. Pageindex uses LLM to read the document structure
3. Build a hierarchical tree index
4. Returns doc_id for all future operations

In [32]:
PDF_PATH = "./data/pdf/sql_interview_guide.pdf"
print(f"Uploading : {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result.get("doc_id")

print("Uploaded")
print(f"document id: {doc_id}")


Uploading : ./data/pdf/sql_interview_guide.pdf
Uploaded
document id: pi-cmsz1dgx000my01p58yxa0q60


In [33]:
print("Building tree index...")
print("This runs once per document - the index is cached for reuse")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")
    
    if status == "completed":
        print("Tree index ready")
        break
    elif status == "failed":
        print("Processing failed. Check pdf format")
        break
    
    time.sleep(3)

Building tree index...
This runs once per document - the index is cached for reuse
Status: completed
Tree index ready


#### Section 3: Inspect the tree structure
Each node has:
- node_id
- title
- page_index
- text
- nodes - child sections

In [34]:
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top level sections: {len(pageindex_tree)}")
print("Raw tree (first node): ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top level sections: 1
Raw tree (first node): 
{
  "title": "Complete SQL Interview Guide for Hard-Level Questions",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Complete SQL Interview Guide for Hard-Level Questions\n\nThis guide is a detailed reference for preparing for hard SQL interview rounds. It covers core SQL foundations, advanced querying patterns, schema design, performance, transactions, concurrency, locking, and common interview-style problems with solutions. The material emphasizes topics frequently highlighted in advanced SQL interview prep resources, especially window functions, CTEs, joins, views, transactions, and locking behavior.[cite:16] [cite:20] [cite:29]\n",
  "text": "# Complete SQL Interview Guide for Hard-Level Questions\n\nThis guide is a detailed reference for preparing for hard SQL interview rounds. It covers core SQL foundations, advanced querying patterns, schema design, performance, transactions, concurrency, locking, and common interview

In [35]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = " " * indent + ("|-- " if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent+1)

print("Full document structure: \n")
print_tree(pageindex_tree)

Full document structure: 

[0000] Complete SQL Interview Guide for Hard-Level Questions (p.1)
 |-- [0001] How to Use This Guide (p.1)
 |-- [0002] SQL Execution Order (p.1)
 |-- [0003] Core Query Building Blocks (p.2)
 |-- [0004] SELECT, WHERE, GROUP BY, HAVING (p.2)
 |-- [0005] DISTINCT (p.2)
 |-- [0006] CASE Expressions (p.2)
 |-- [0007] Joins (p.2)
 |-- [0008] Inner Join (p.3)
 |-- [0009] Left Join (p.3)
 |-- [0010] Anti-Join Pattern (p.3)
 |-- [0011] Self Join (p.3)
 |-- [0012] Non-Equi and Range Joins (p.4)
 |-- [0013] Subqueries (p.4)
 |-- [0014] Scalar Subquery (p.4)
 |-- [0015] Correlated Subquery (p.4)
 |-- [0016] Aggregations (p.5)
 |-- [0017] COUNT Variants (p.5)
 |-- [0018] Conditional Aggregation (p.5)
 |-- [0019] Pivot-Like Aggregation (p.5)
 |-- [0020] Window Functions (p.6)
 |-- [0021] ROW_NUMBER, RANK, DENSE_RANK (p.6)
 |-- [0022] Running Total (p.6)
 |-- [0023] Moving Average (p.7)
 |-- [0024] LAG and LEAD (p.7)
 |-- [0025] FIRST_VALUE and LAST_VALUE (p.7)
 |-- [0026] 

In [36]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print("Each node = one retrieval section of the document")

Total nodes in tree: 102
Each node = one retrieval section of the document


#### Section 4: LLM Tree Search - The core of pageindex

This is where PageIndex fundamentally differs from vector RAG.

- Vector RAG retrieval:
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
Problem: finds what's similar, not what's relevant

- PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [37]:
import json
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from langchain_core.language_models import BaseChatModel


class TreeSearchResponse(BaseModel):
    thinking: str = Field(description="Step-by-step reasoning about which sections are relevant")
    node_list: List[str] = Field(description="List of node IDs that likely contain the answer")
    

def llm_tree_search(query: str, tree: list, llm: BaseChatModel) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
                Your task: identify which node IDs most likely contain the answer to the query.
                Think step-by-step about which sections are relevant.

                Query: {query}

                Document Tree:
                {json.dumps(compressed_tree, indent=2)}

                Reply ONLY with a JSON object matching this schema:
                {{
                "thinking": "<your step-by-step reasoning>",
                "node_list": ["node_id1", "node_id2"]
                }}"""

    # Wrap LLM to enforce JSON schema
    structured_llm = llm.with_structured_output(TreeSearchResponse, method="json_schema")
    
    # Invoke and get parsed Pydantic model
    response: TreeSearchResponse = structured_llm.invoke(prompt)
    
    # Convert to dict if needed
    return response.model_dump()

In [39]:
query = "what are views in sql ?"
print(f"Query: {query}\n")
result = llm_tree_search(query, pageindex_tree, llm)

print("LLM reasoning:")
print(result.get("thinking", "N/A"))
print()

print(f"Selected node ids: {result.get("node_list", [])}")

Query: what are views in sql ?

LLM reasoning:
The user is asking for a definition of 'views' in SQL. I need to find nodes in the document tree that discuss SQL views. 

1. I will look for keywords like 'view', 'views', 'CREATE VIEW'.
2. Node '0030' has the title 'Views and Materialized Views' and mentions 'Views provide reusable query abstractions'. This is highly relevant.
3. Node '0031' is titled 'View' and provides a `CREATE VIEW` example. This is also highly relevant.
4. Node '0033' compares 'Temporary Tables vs CTEs vs Views', which would likely define what a view is in comparison to other concepts.
5. Node '0063' is titled 'Views, Permissions, and Security Basics' and mentions that 'Views can simplify access control'. This suggests it will define what views are and their purpose.

Therefore, nodes 0030, 0031, 0033, and 0063 are the most likely to contain the answer.

Selected node ids: ['0030', '0031', '0033', '0063']


#### Section 5: Full End to End RAG Pipeline

3 steps:

- Tree Search → LLM picks relevant node_ids
- Retrieve → Fetch the actual section content from those nodes
- Generate → LLM writes a grounded answer with page citations

What makes this better than vector RAG:
Retrieved content has titles + page numbers (traceable)
LLM can cite exactly which section the answer comes from
No hallucination from irrelevant chunks

In [40]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [41]:
def generate_answer(query: str, nodes: list, llm: BaseChatModel) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
                Answer the question using ONLY the provided context.
                For every claim you make, cite the section title and page number in parentheses.
                Be concise and precise.

                Question: {query}

                Context:
                {context}

                Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

In [42]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree, llm)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes, llm)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [43]:
answer = vectorless_rag(query="what are views in sql ?", tree=pageindex_tree)

🔍 Query: what are views in sql ?

🧠 Reasoning: The user is asking for a definition of 'views' in SQL. I need to find nodes that discuss SQL views. Node '0030' has the title 'Views and Materialized Views' and its summary mentions 'Views provide reu...
🎯 Retrieved node IDs: ['0030', '0031', '0063', '0080']
📄 Sections found: ['Views and Materialized Views', 'View', 'Views, Permissions, and Security Basics', 'Create a View for Active Premium Customers']

📝 Answer:
Views provide reusable query abstractions (Views and Materialized Views, 9). They can simplify access control by exposing a restricted projection of base tables (Views, Permissions, and Security Basics, 17). An example of a view is `active_customers`, which selects `customer_id` and `customer_name` from the `customers` table where the `status` is 'active' (View, 9). Another example is `active_premium_customers`, which selects `customer_id`, `customer_name`, and `email` from `customers` where the `status` is 'active' and the `tier

#### Section 6: Expert Guided Retrieval
The killer feature no one talks about.

With vector RAG, injecting domain expertise requires fine-tuning the embedding model — expensive and time-consuming.

With PageIndex, you just add rules to the prompt:

"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.

#### Section 7: Zero LLM Setup
When to use this:

- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and doc_id.

In [44]:
question = "What are the key findings in this document ?"
response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]['message']['content']
print("Chat API answer: ")
print(answer)

Chat API answer: 
Here's a summary of the key findings and topics covered in the **Complete SQL Interview Guide for Hard-Level Questions**:

---

## 🔑 Key Findings

### 1. **What Hard SQL Interviews Really Test**
Interviews assess three things simultaneously: **correctness**, **performance awareness**, and the ability to **explain trade-offs** — not just syntax memorization.

### 2. **SQL Execution Order is Critical**
Many candidates fail because they don't understand logical execution order:
> `FROM/JOIN → WHERE → GROUP BY → HAVING → Window Functions → SELECT → DISTINCT → ORDER BY → LIMIT`

### 3. **Core Query Building Blocks Must Be Flawless**
- **WHERE vs. HAVING**: A common interview trap — `WHERE` filters before aggregation, `HAVING` filters after.
- **DISTINCT**: Often misused; the real issue is usually an incorrect join.
- **CASE expressions**: Essential for conditional bucketing, scoring, and manual pivoting.

### 4. **Joins Are Central to Interview Success**
The guide covers:


In [46]:
# Multi turn conversation

conversation_history = []
def chat_with_doc(user_message: str, doc_id: str) -> str:
    global conversation_history
    conversation_history.append({"role": "user", "content": user_message})
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply

question = "how to find the second largest salaried employee from sql table ?"
reply = chat_with_doc(question, doc_id)
print(f"\nUser: {question}")
print(f"\nAssistant: {reply[:400]}")


User: how to find the second largest salaried employee from sql table ?

Assistant: Found it on **page 8** of your guide! Here are the two approaches:

---

## 🥈 Finding the Second Highest Salary

### ✅ Method 1: Subquery with MAX (Simple & Classic)
```sql
SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
```
**How it works:**
- The inner query finds the **highest salary**.
- The outer query finds the **MAX salary** tha


#### Section 8: Self Hosted Open Source Platform
Use this when:

- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic
- The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

What the CLI does:

- Reads your PDF
- Detects existing Table of Contents (if any)
- Uses GPT-4o to build the hierarchical tree
- Saves a document_name_pageindex.json alongside your PDF

#### Section 9: Vector RAG vs PageIndex

##### When to use which
Use Vector RAG when:

- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important
- You need sub-second retrieval on millions of documents

Use PageIndex when:

- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure

#### Section 10: Cleanup
Delete documents from the page index cloud when you are done to keep your storage clean

In [47]:
pi_client.delete_document(doc_id)
print(f"Deleted document: {doc_id}")

Deleted document: pi-cmsz1dgx000my01p58yxa0q60
